<a href="https://colab.research.google.com/github/CalculatedContent/xgboost2ww/blob/main/notebooks/XGBoost2WWAdultIncomeHyperparameterSweep.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# xgboost2ww Adult Income Hyperparameter Sweep (Checkpointable)

This notebook mirrors the Adult Income example, but trains a **set of well-regularized XGBoost models** with different hyperparameters.

It is designed to be restart-safe:
- save per-model results to Google Drive checkpoints,
- resume incomplete runs,
- and preserve experiment config metadata.

## 1) Mount Google Drive and configure checkpoint paths

In [ ]:
from pathlib import Path
from datetime import datetime
import json
import os

IN_COLAB = False
try:
    from google.colab import drive
    IN_COLAB = True
except Exception:
    IN_COLAB = False

# ===== USER CONFIG =====
RESTART_EXPERIMENT = True
EXPERIMENT_NAME = "adult_income_hparam_sweep_run01"  # required if RESTART_EXPERIMENT=True
DEFAULT_EXPERIMENT_BASENAME = "adult_income_hparam_sweep"
AUTO_INCREMENT_IF_NAME_MISSING = True

# Colab Drive root (used when IN_COLAB=True)
EXPERIMENT_ROOT = Path("/content/drive/MyDrive/xgbwwdata/experiment_checkpoints")

# Local fallback root (used when running outside Colab)
LOCAL_EXPERIMENT_ROOT = Path("./experiment_checkpoints")

CHECKPOINT_SAVE_EVERY = 1
# =======================


def next_experiment_name(root: Path, base_name: str) -> str:
    existing = [d.name for d in root.glob(f"{base_name}_run*") if d.is_dir()]
    nums = []
    for name in existing:
        suffix = name.replace(f"{base_name}_run", "")
        if suffix.isdigit():
            nums.append(int(suffix))
    n = (max(nums) + 1) if nums else 1
    return f"{base_name}_run{n:02d}"


if IN_COLAB:
    drive.mount("/content/drive")
    ROOT = EXPERIMENT_ROOT
else:
    ROOT = LOCAL_EXPERIMENT_ROOT
    print("Not running in Colab; using local checkpoint folder:", ROOT.resolve())

ROOT.mkdir(parents=True, exist_ok=True)

if RESTART_EXPERIMENT:
    if not EXPERIMENT_NAME:
        raise ValueError("Set EXPERIMENT_NAME when RESTART_EXPERIMENT=True")
    EXPERIMENT_ID = EXPERIMENT_NAME
else:
    if EXPERIMENT_NAME:
        EXPERIMENT_ID = EXPERIMENT_NAME
    elif AUTO_INCREMENT_IF_NAME_MISSING:
        EXPERIMENT_ID = next_experiment_name(ROOT, DEFAULT_EXPERIMENT_BASENAME)
    else:
        EXPERIMENT_ID = DEFAULT_EXPERIMENT_BASENAME

CHECKPOINT_DIR = ROOT / EXPERIMENT_ID
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CHECKPOINT_RESULTS_CSV = CHECKPOINT_DIR / "checkpoint_results.csv"
EXPERIMENT_CONFIG_JSON = CHECKPOINT_DIR / "experiment_config.json"

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Checkpoint results:", CHECKPOINT_RESULTS_CSV)

## 2) Install required packages (Colab)

In [ ]:
!pip -q install xgboost weightwatcher xgboost2ww

## 3) Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import sparse
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_openml
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import xgboost as xgb
import weightwatcher as ww
from xgboost2ww import convert

## 4) Reproducibility + shared settings

In [ ]:
RNG = 123
np.random.seed(RNG)

TEST_SIZE = 0.2
NFOLDS = 5
T_POINTS = 80
MAX_DENSE_ELEMENTS = int(2e8)  # safety if convert needs dense input

## 5) Load Adult Income data and preprocess

In [ ]:
raw = fetch_openml(data_id=1590, as_frame=True)
df = raw.frame.copy()

y = (df["class"] == ">50K").astype(int).to_numpy()
X_df = df.drop(columns=["class"])

categorical_cols = X_df.select_dtypes(include=["object", "category"]).columns
numeric_cols = X_df.select_dtypes(exclude=["object", "category"]).columns

pre = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", "passthrough", numeric_cols),
    ]
)

X = pre.fit_transform(X_df).astype(np.float32)

idx_train, idx_test = train_test_split(
    np.arange(len(y)), test_size=TEST_SIZE, random_state=RNG, stratify=y
)

Xtr, Xte = X[idx_train], X[idx_test]
ytr, yte = y[idx_train], y[idx_test]

if sparse.issparse(Xtr):
    scaler = StandardScaler(with_mean=False)
else:
    scaler = StandardScaler()

Xtr = scaler.fit_transform(Xtr).astype(np.float32)
Xte = scaler.transform(Xte).astype(np.float32)

print("Train shape:", Xtr.shape)
print("Test shape:", Xte.shape)
print("Positive class fraction:", y.mean().round(4))

## 6) Define a controlled hyperparameter sweep (good -> mildly worse)

In [ ]:
base = dict(
    objective="binary:logistic",
    eval_metric="logloss",
    tree_method="hist",
    seed=RNG,
)

configs = [
    dict(name="A_strong_reg", max_depth=3, learning_rate=0.04, subsample=0.80, colsample_bytree=0.75,
         reg_lambda=8.0, reg_alpha=1.0, min_child_weight=8.0, gamma=1.5, num_boost_round=900, early_stopping_rounds=80),
    dict(name="B_balanced_reg", max_depth=3, learning_rate=0.05, subsample=0.80, colsample_bytree=0.80,
         reg_lambda=5.0, reg_alpha=0.5, min_child_weight=5.0, gamma=1.0, num_boost_round=700, early_stopping_rounds=60),
    dict(name="C_reference", max_depth=3, learning_rate=0.06, subsample=0.85, colsample_bytree=0.85,
         reg_lambda=3.5, reg_alpha=0.25, min_child_weight=4.0, gamma=0.8, num_boost_round=600, early_stopping_rounds=50),
    dict(name="D_slightly_richer", max_depth=4, learning_rate=0.06, subsample=0.85, colsample_bytree=0.85,
         reg_lambda=3.0, reg_alpha=0.2, min_child_weight=3.0, gamma=0.5, num_boost_round=500, early_stopping_rounds=40),
    dict(name="E_mildly_underfit", max_depth=2, learning_rate=0.03, subsample=0.75, colsample_bytree=0.70,
         reg_lambda=10.0, reg_alpha=1.5, min_child_weight=10.0, gamma=2.0, num_boost_round=1100, early_stopping_rounds=90),
    dict(name="F_higher_lr", max_depth=3, learning_rate=0.10, subsample=0.80, colsample_bytree=0.80,
         reg_lambda=4.0, reg_alpha=0.2, min_child_weight=4.0, gamma=0.6, num_boost_round=350, early_stopping_rounds=30),
]

configs_df = pd.DataFrame(configs)
configs_df

## 7) Save experiment config (for reproducibility + restart visibility)

In [ ]:
experiment_config = {
    "created_utc": datetime.utcnow().isoformat() + "Z",
    "experiment_id": EXPERIMENT_ID,
    "restart_experiment": RESTART_EXPERIMENT,
    "rng": RNG,
    "test_size": TEST_SIZE,
    "nfolds": NFOLDS,
    "t_points": T_POINTS,
    "max_dense_elements": MAX_DENSE_ELEMENTS,
    "checkpoint_results_csv": str(CHECKPOINT_RESULTS_CSV),
    "num_configs": len(configs),
    "config_names": [c["name"] for c in configs],
    "configs": configs,
    "base_params": base,
}

with open(EXPERIMENT_CONFIG_JSON, "w") as f:
    json.dump(experiment_config, f, indent=2)

print("Saved:", EXPERIMENT_CONFIG_JSON)

## 8) Load checkpoint (if present) and identify remaining configs

In [ ]:
if CHECKPOINT_RESULTS_CSV.exists():
    checkpoint_df = pd.read_csv(CHECKPOINT_RESULTS_CSV)
    print(f"Loaded checkpoint with {len(checkpoint_df)} rows")
else:
    checkpoint_df = pd.DataFrame()
    print("No checkpoint found yet. Starting fresh.")

completed = set(checkpoint_df.get("model", pd.Series(dtype=str)).dropna().astype(str).tolist())
pending_configs = [c for c in configs if c["name"] not in completed]

print("Completed models:", sorted(completed) if completed else "None")
print("Pending models:", [c["name"] for c in pending_configs])

## 9) Train pending models, checkpoint each result, and resume safely

In [ ]:
def safe_convert_to_w7(model, X_train, y_train, params, best_round):
    X_for_convert = X_train
    try:
        return convert(
            model=model,
            data=X_for_convert,
            labels=y_train,
            W="W7",
            nfolds=NFOLDS,
            t_points=T_POINTS,
            random_state=RNG,
            train_params=params,
            num_boost_round=best_round,
            multiclass="error",
            return_type="torch",
            verbose=False,
        )
    except Exception as e:
        print(f"convert() sparse attempt failed: {type(e).__name__}: {e}")
        if sparse.issparse(X_train):
            dense_cost = int(X_train.shape[0]) * int(X_train.shape[1])
            if dense_cost > MAX_DENSE_ELEMENTS:
                raise MemoryError(
                    f"Refusing to densify X_train with shape={X_train.shape}, elements={dense_cost:,}"
                )
            X_for_convert = X_train.toarray().astype(np.float32)
            return convert(
                model=model,
                data=X_for_convert,
                labels=y_train,
                W="W7",
                nfolds=NFOLDS,
                t_points=T_POINTS,
                random_state=RNG,
                train_params=params,
                num_boost_round=best_round,
                multiclass="error",
                return_type="torch",
                verbose=False,
            )
        raise


dtr = xgb.DMatrix(Xtr, label=ytr)
dte = xgb.DMatrix(Xte, label=yte)

running_results = checkpoint_df.copy() if not checkpoint_df.empty else pd.DataFrame()
new_rows_buffer = []

for i, cfg in enumerate(pending_configs, start=1):
    print(f"[{i}/{len(pending_configs)}] Training {cfg['name']}...")

    run_cfg = cfg.copy()
    num_boost_round = int(run_cfg.pop("num_boost_round"))
    early_stopping_rounds = int(run_cfg.pop("early_stopping_rounds"))
    model_name = run_cfg.pop("name")

    params = {**base, **run_cfg}

    bst = xgb.train(
        params=params,
        dtrain=dtr,
        num_boost_round=num_boost_round,
        evals=[(dtr, "train"), (dte, "test")],
        early_stopping_rounds=early_stopping_rounds,
        verbose_eval=False,
    )

    best_round = int(bst.best_iteration + 1)

    p_tr = bst.predict(dtr)
    p_te = bst.predict(dte)
    train_acc = float(accuracy_score(ytr, (p_tr > 0.5).astype(int)))
    test_acc = float(accuracy_score(yte, (p_te > 0.5).astype(int)))
    gap = float(train_acc - test_acc)

    train_ll = float(log_loss(ytr, p_tr))
    test_ll = float(log_loss(yte, p_te))

    layer_w7 = safe_convert_to_w7(bst, Xtr, ytr, params, best_round)
    watcher = ww.WeightWatcher(model=layer_w7)
    ww_df = watcher.analyze(randomize=True, detX=True)

    alpha = float(ww_df["alpha"].iloc[0]) if "alpha" in ww_df.columns else np.nan
    erg_gap = float(ww_df["ERG_gap"].iloc[0]) if "ERG_gap" in ww_df.columns else np.nan

    row = {
        "model": model_name,
        "status": "completed",
        "best_round": best_round,
        "train_acc": train_acc,
        "test_acc": test_acc,
        "generalization_gap": gap,
        "train_logloss": train_ll,
        "test_logloss": test_ll,
        "alpha": alpha,
        "ERG_gap": erg_gap,
    }
    row.update(run_cfg)
    new_rows_buffer.append(row)

    print(f"  done: train_acc={train_acc:.4f}, test_acc={test_acc:.4f}, gap={gap:.4f}, alpha={alpha:.3f}, ERG_gap={erg_gap:.3f}")

    if (i % CHECKPOINT_SAVE_EVERY == 0) or (i == len(pending_configs)):
        add_df = pd.DataFrame(new_rows_buffer)
        running_results = pd.concat([running_results, add_df], ignore_index=True)
        running_results = running_results.drop_duplicates(subset=["model"], keep="last")
        running_results.to_csv(CHECKPOINT_RESULTS_CSV, index=False)
        print(f"  checkpoint saved ({len(running_results)} total rows) -> {CHECKPOINT_RESULTS_CSV}")
        new_rows_buffer = []

## 10) Load final results table from checkpoint

In [ ]:
results_df = pd.read_csv(CHECKPOINT_RESULTS_CSV)
results_df = results_df.sort_values("test_acc", ascending=False).reset_index(drop=True)
results_df

## 11) Quick quality guardrail (no severe overfit)

In [ ]:
max_gap = pd.to_numeric(results_df["generalization_gap"], errors="coerce").max()
print(f"Max generalization gap across sweep: {max_gap:.4f}")

if max_gap > 0.08:
    print("⚠️ Some models may be overfitting more than intended.")
else:
    print("✅ Sweep looks reasonably controlled (no severe overfitting).")

## 12) Compare WW metrics to accuracy + gap

In [ ]:
display_cols = [
    "model", "status", "best_round", "alpha", "ERG_gap", "train_acc", "test_acc", "generalization_gap",
    "max_depth", "learning_rate", "subsample", "colsample_bytree", "reg_lambda", "reg_alpha",
    "min_child_weight", "gamma"
]
results_df[display_cols]

In [ ]:
corr_cols = ["alpha", "ERG_gap", "train_acc", "test_acc", "generalization_gap"]
results_df[corr_cols] = results_df[corr_cols].apply(pd.to_numeric, errors="coerce")
results_df[corr_cols].corr(method="spearman")

## 13) Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

axes[0, 0].scatter(results_df["alpha"], results_df["test_acc"], s=60)
axes[0, 0].set_xlabel("alpha")
axes[0, 0].set_ylabel("test_acc")
axes[0, 0].set_title("alpha vs test accuracy")

axes[0, 1].scatter(results_df["alpha"], results_df["generalization_gap"], s=60)
axes[0, 1].set_xlabel("alpha")
axes[0, 1].set_ylabel("generalization_gap")
axes[0, 1].set_title("alpha vs generalization gap")

axes[1, 0].scatter(results_df["ERG_gap"], results_df["test_acc"], s=60)
axes[1, 0].set_xlabel("ERG_gap")
axes[1, 0].set_ylabel("test_acc")
axes[1, 0].set_title("ERG_gap vs test accuracy")

axes[1, 1].scatter(results_df["ERG_gap"], results_df["generalization_gap"], s=60)
axes[1, 1].set_xlabel("ERG_gap")
axes[1, 1].set_ylabel("generalization_gap")
axes[1, 1].set_title("ERG_gap vs generalization gap")

for _, r in results_df.iterrows():
    axes[0, 0].annotate(r["model"], (r["alpha"], r["test_acc"]), fontsize=8)
    axes[0, 1].annotate(r["model"], (r["alpha"], r["generalization_gap"]), fontsize=8)
    axes[1, 0].annotate(r["model"], (r["ERG_gap"], r["test_acc"]), fontsize=8)
    axes[1, 1].annotate(r["model"], (r["ERG_gap"], r["generalization_gap"]), fontsize=8)

plt.tight_layout()
plt.show()

## 14) Save final results dataframe to Google Drive (or local checkpoint dir)

In [ ]:
FINAL_RESULTS_CSV = CHECKPOINT_DIR / "adult_hyperparameter_sweep_final.csv"
results_df.to_csv(FINAL_RESULTS_CSV, index=False)

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results checkpoint:", CHECKPOINT_RESULTS_CSV)
print("Final results dataframe saved to:", FINAL_RESULTS_CSV)
print("Config file:", EXPERIMENT_CONFIG_JSON)


## 15) Final requested plots: accuracy/gap vs alpha

In [ ]:
plot_df = results_df.copy()
for c in ["alpha", "train_acc", "test_acc", "generalization_gap"]:
    plot_df[c] = pd.to_numeric(plot_df[c], errors="coerce")

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(plot_df["alpha"], plot_df["test_acc"], s=70)
axes[0].set_xlabel("alpha")
axes[0].set_ylabel("test accuracy")
axes[0].set_title("Test accuracy vs alpha")

axes[1].scatter(plot_df["alpha"], plot_df["train_acc"], s=70)
axes[1].set_xlabel("alpha")
axes[1].set_ylabel("training accuracy")
axes[1].set_title("Training accuracy vs alpha")

axes[2].scatter(plot_df["alpha"], plot_df["generalization_gap"], s=70)
axes[2].set_xlabel("alpha")
axes[2].set_ylabel("generalization gap")
axes[2].set_title("Generalization gap vs alpha")

for _, r in plot_df.iterrows():
    axes[0].annotate(r["model"], (r["alpha"], r["test_acc"]), fontsize=8)
    axes[1].annotate(r["model"], (r["alpha"], r["train_acc"]), fontsize=8)
    axes[2].annotate(r["model"], (r["alpha"], r["generalization_gap"]), fontsize=8)

plt.tight_layout()
plt.show()